In [12]:
# if needed
# !pip install xgboost scikit-learn pandas joblib

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBClassifier, XGBRegressor
import joblib

In [13]:
df = pd.read_csv("auction_dataset.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (18000, 12)


,auction_id,starting_price,current_bid,bidders,total_bids,avg_increment,bid_velocity,auction_duration,time_remaining,final_price,winner_id,will_win
0,1,811,2338,9,42,82,8.0,58,70,2350,17,1
1,2,1906,3024,20,104,193,8.0,180,3,6921,5,0
2,3,1490,4820,2,36,151,8.0,192,51,4622,16,1
3,4,1237,3725,9,44,125,8.0,115,41,3680,1,1
4,5,396,1099,11,57,40,8.0,110,76,1383,8,1


In [14]:
print("Columns:\n")
print(df.columns)

Columns:

Index(['auction_id', 'starting_price', 'current_bid', 'bidders', 'total_bids',
       'avg_increment', 'bid_velocity', 'auction_duration', 'time_remaining',
       'final_price', 'winner_id', 'will_win'],
      dtype='object')


In [ ]:
# Regression target
y_reg = df["final_price"]

# Classification target (auction success)
df["auction_winner"] = (df["final_price"] > df["final_price"].median()).astype(int)

y_cls = df["auction_winner"]

print("Class Distribution:")
print(y_cls.value_counts())

Class Distribution:
auction_winner
0    9002
1    8998
Name: count, dtype: int64


In [ ]:
# Remove target columns to avoid data leakage

X = df.drop(["final_price", "auction_winner"], axis=1)

print("Features used for training:")
print(X.columns)

Features used for training:
Index(['auction_id', 'starting_price', 'current_bid', 'bidders', 'total_bids',
       'avg_increment', 'bid_velocity', 'auction_duration', 'time_remaining',
       'winner_id', 'will_win'],
      dtype='object')


In [17]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X,
    y_reg,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.model_selection import train_test_split

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X,
    y_cls,
    test_size=0.2,
    random_state=42,
    stratify=y_cls
)

In [19]:
scaler = StandardScaler()

X_train_reg = scaler.fit_transform(X_train_reg)
X_test_reg = scaler.transform(X_test_reg)

X_train_cls = scaler.fit_transform(X_train_cls)
X_test_cls = scaler.transform(X_test_cls)

In [20]:
reg_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

reg_model.fit(X_train_reg, y_train_reg)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None
,feature_types,None


In [21]:
y_pred_reg = reg_model.predict(X_test_reg)

mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)

print("Regression Evaluation")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

Regression Evaluation
MAE: 146.18182373046875
RMSE: 211.51109050413882
R2 Score: 0.9879304766654968


In [22]:
cls_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

cls_model.fit(X_train_cls, y_train_cls)

,objective,'binary:logistic'
,use_label_encoder,None
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [23]:
y_pred_cls = cls_model.predict(X_test_cls)

print("Accuracy:", accuracy_score(y_test_cls, y_pred_cls))

print("\nClassification Report:\n")
print(classification_report(y_test_cls, y_pred_cls))

Accuracy: 0.9738888888888889

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.97      0.97      1800
           1       0.97      0.97      0.97      1800

    accuracy                           0.97      3600
   macro avg       0.97      0.97      0.97      3600
weighted avg       0.97      0.97      0.97      3600



In [24]:
train_acc = cls_model.score(X_train_cls, y_train_cls)
test_acc = cls_model.score(X_test_cls, y_test_cls)

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

Train Accuracy: 0.9940972222222222
Test Accuracy: 0.9738888888888889


In [25]:
scores = cross_val_score(cls_model, X, y_cls, cv=5)

print("Cross Validation Scores:", scores)
print("Average CV Score:", scores.mean())

Cross Validation Scores: [0.97611111 0.97055556 0.97361111 0.97944444 0.9725    ]
Average CV Score: 0.9744444444444443


In [26]:
joblib.dump(cls_model, "auction_win_probability_model.pkl")
joblib.dump(reg_model, "auction_final_price_model.pkl")

print("Models saved successfully")

Models saved successfully


In [27]:
win_model = joblib.load("auction_win_probability_model.pkl")
price_model = joblib.load("auction_final_price_model.pkl")

In [28]:
sample = X_test_cls[0].reshape(1,-1)

win_probability = win_model.predict_proba(sample)[0][1]
predicted_price = price_model.predict(sample)[0]

print("Win Probability:", win_probability)
print("Predicted Final Price:", predicted_price)

Win Probability: 0.99997234
Predicted Final Price: 7509.1523
